# CMPE 401 Instructor-Defined Project 1
## YOLOv11 Object Detection Baseline Training & Analysis

This notebook covers:

- Training YOLOv11 as a baseline model on the VisDrone-DET trainset dataset.
- Loss curve and fitting analysis (overfitting/underfitting diagnosis)
- One round of a controlled experiment to compare against the baseline model.
- Iterative model improvement to make the model more accurate and precise.

---

### Dataset: VisDrone2019-DET-train
- 6,471 drone-captured images  
- Images are of dense scenes with small objects
- 11 object categories (ignored regions excluded)

### Model: YOLOv11 nano (YOLOv11n)
The nano model is used as the baseline because it trains fastest and sets a lower-bound reference point for later model capacity experiments.


## Environment Config & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/YOLOv11-Project') # results are saved here
DRIVE_DIR.mkdir(exist_ok=True)

# unzip the directory containing the dataset
ZIP_PATH = '/content/drive/MyDrive/VisDrone2019-DET-train.zip'

if not Path('/content/VisDrone2019-DET-train/images').exists():
  print('Unzipping dataset')
  with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/')
  n = len(list(Path('/content/VisDrone2019-DET-train/images').glob('*.jpg')))
  print(f'Done: {n} images extracted.')
else: print('Training dataset already extracted.')

In [ ]:
%pip install ultralytics numpy matplotlib pandas Pillow tqdm -q

In [ ]:
import os
import shutil
import random
import csv
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

SEED = 15
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = DRIVE_DIR # save data we need to persist to Google Drive

# Datasets were extracted to local /content/ storage
RAW_DIR     = Path('/content/VisDrone2019-DET-train')
RAW_IMAGES  = RAW_DIR  / 'images'
RAW_ANNOTS  = RAW_DIR  / 'annotations'

# YOLO-formatted dataset will be saved in Drive
DATASET_DIR = BASE_DIR / 'dataset'
TRAIN_IMG   = DATASET_DIR / 'images'  / 'train'
VAL_IMG     = DATASET_DIR / 'images'  / 'val'
TRAIN_LBL   = DATASET_DIR / 'labels'  / 'train'
VAL_LBL     = DATASET_DIR / 'labels'  / 'val'

RUNS_DIR    = BASE_DIR / 'runs'

BASELINE_NAME = 'yolov11n_baseline' # baseline model
EXP_NAME = 'yolov11n_1280px'        # experiment model
IMP_NAME1 = 'yolov11n_improvement1' # improvement cycle 1 model
IMP_NAME2 = 'yolov11n_improvement2' # improvement cycle 2 model

FIG_DPI = 130 # for graphing
plt.style.use('seaborn-v0_8-whitegrid') # add grid to graphs

print('Project root (Drive) :', BASE_DIR)
print('Raw images  (local)  :', len(list(RAW_IMAGES.glob('*.jpg'))), 'files')
print('Annotations (local)  :', len(list(RAW_ANNOTS.glob('*.txt'))), 'files')


## Data Prep and Cleaning

### Convert VisDrone Dataset Annotations to YOLO Format

#### VisDrone annotation format column names
```
<bbox_left>, <bbox_top>, <bbox_width>, <bbox_height>, <score>, <category>, <truncation>, <occlusion>
```

#### YOLO label format column names
```
<class_id>  <x_center_norm>  <y_center_norm>  <w_norm>  <h_norm>
```

**Category mapping**  
VisDrone category `0` is an `ignored region`. These are skipped.

Categories `1-11` are remapped to YOLO class IDs `0-10`.

| VisDrone ID | Class Name         | YOLO ID |
|-------------|--------------------|---------|
| 1           | pedestrian         | 0       |
| 2           | people             | 1       |
| 3           | bicycle            | 2       |
| 4           | car                | 3       |
| 5           | van                | 4       |
| 6           | truck              | 5       |
| 7           | tricycle           | 6       |
| 8           | awning-tricycle    | 7       |
| 9           | bus                | 8       |
| 10          | motor              | 9       |
| 11          | others             | 10      |

In [ ]:
CLASS_NAMES = [
    'pedestrian', 'people', 'bicycle', 'car', 'van',
    'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor', 'others'
]
NUM_CLASSES = len(CLASS_NAMES)

def visdrone_to_yolo(annot_path: Path, img_w: int, img_h: int) -> list[str]:
    """
    Convert a VisDrone annotation file to a YOLO-compatible format.
    """
    lines = []
    with open(annot_path) as f:
        for row in csv.reader(f):
            if len(row) < 6:
                continue
            bx, by, bw, bh = int(row[0]), int(row[1]), int(row[2]), int(row[3])
            cat = int(row[5])
            if cat == 0 or cat > 11: # skip 'ignored regions' and unknowns
                continue
            if bw <= 0 or bh <= 0:# invalid box coords
                continue
            yolo_id = cat - 1 # remap 1-11 to 0-10
            cx = (bx + bw / 2) / img_w
            cy = (by + bh / 2) / img_h
            nw = bw / img_w
            nh = bh / img_h

            # clamp box coords to [0, 1]
            cx, cy = min(max(cx, 0), 1), min(max(cy, 0), 1)
            nw, nh = min(max(nw, 0), 1), min(max(nh, 0), 1)
            lines.append(f'{yolo_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')

    return lines

print('Conversion function defined.')
print(f'Classes ({NUM_CLASSES}):', CLASS_NAMES)

In [ ]:
# Split the dataset into 80% for training and 20% for validation.
VAL_SPLIT = 0.20

all_images = sorted(RAW_IMAGES.glob('*.jpg'))
random.shuffle(all_images)
n_val   = int(len(all_images) * VAL_SPLIT)
n_train = len(all_images) - n_val

val_images   = set(all_images[:n_val])
train_images = set(all_images[n_val:])

print(f'Total images : {len(all_images)}')
print(f'Training     : {n_train}')
print(f'Validation   : {n_val}')

In [ ]:
# Make the directory structure for the converted dataset and save it to Drive
for d in [TRAIN_IMG, VAL_IMG, TRAIN_LBL, VAL_LBL]:
    d.mkdir(parents=True, exist_ok=True)

skipped = 0
converted = {'train': 0, 'val': 0, 'test': 0}

for img_path in tqdm(all_images, desc='Converting & splitting train/val'):
    annot_path = RAW_ANNOTS / (img_path.stem + '.txt')
    if not annot_path.exists():
        skipped += 1
        continue

    # grab each image's dimensions without loading full pixel data
    with Image.open(img_path) as im:
        img_w, img_h = im.size

    is_val = img_path in val_images
    dst_img_dir = VAL_IMG   if is_val else TRAIN_IMG
    dst_lbl_dir = VAL_LBL   if is_val else TRAIN_LBL
    split_key   = 'val'     if is_val else 'train'

    # copy the image
    dst_img = dst_img_dir / img_path.name
    if not dst_img.exists():
        shutil.copy2(img_path, dst_img)

    # convert the annotation file to the YOLO format and save it to a new .txt file
    dst_lbl = dst_lbl_dir / (img_path.stem + '.txt')
    if not dst_lbl.exists():
        yolo_lines = visdrone_to_yolo(annot_path, img_w, img_h)
        dst_lbl.write_text('\n'.join(yolo_lines))

    converted[split_key] += 1

print(f"\nDone. Train: {converted['train']}  Val: {converted['val']}  Skipped: {skipped}")

### Create `dataset.yaml` config file

In [ ]:

yaml_content = f"""# VisDrone-DET YOLO-compatible format
path: {DATASET_DIR} # dataset's root
train: images/train
val:   images/val

nc: {NUM_CLASSES} # 11
names: {CLASS_NAMES} # e.g., pedestrian, car, etc.
"""

yaml_path = DATASET_DIR / 'dataset.yaml'
yaml_path.write_text(yaml_content)
print(yaml_content)


## Baseline Model Training

### Baseline Hyperparameters

The model's training has been optimized for an Nvidia A100 GPU:

| Hyperparameter | Value | Note |
|---|---|---|
| Epochs | 50 | Should be enough to observe convergence trends |
| Image size | 640 | Standard YOLO res |
| Batch size | 64 / 128 | Dependent on the GPU's RAM. <br>For models with 40GB, images are processed in batches of 64 at a time. <br>For models with 80GB, 128 are processed at a time. |
| Workers | 8 | The number of worker threads on the CPU that pre-load and pre-process <br>the next batch of images while the GPU is training off of the current batch. |
| Cache | RAM | All images are initially cached into RAM so that they don't have to be <br>reloaded at the start of each epoch. |
| Automatic Mixed Precision (AMP) | `True` | Most values are used and stored as 16-bit floats instead of 32-bit because <br>A100 tensor cores are optimized for 16-bit (this gives roughly 2x the <br>throughput). Some critical values (e.g., loss scaling) are automatically kept <br>as 32-bit floats to prevent them from underflowing or overflowing. |


In [ ]:
import torch
from ultralytics import YOLO

gpu  = torch.cuda.get_device_properties(0)
vram = gpu.total_memory / 1e9
print(f'GPU  : {gpu.name}')
print(f'VRAM : {vram:.1f} GB')
print(f'CUDA : {torch.version.cuda}')

BATCH_SIZE = 64 if vram < 70 else 128
print(f'Batch: {BATCH_SIZE}  (auto-selected for {vram:.0f} GB VRAM)')

model = YOLO('yolo11n.pt') # Load pretrained weights

results = model.train(
    data     = str(yaml_path),
    epochs   = 50,
    imgsz    = 640,
    batch    = BATCH_SIZE,
    workers  = 8,     # parallel dataloaders
    cache    = 'ram', # load all images into RAM once at the start
    amp      = True,  # Automatic Mixed Precision
    cos_lr   = True,  # cosine LR decay
    name     = BASELINE_NAME,
    project  = str(RUNS_DIR),
    seed     = SEED,
    verbose  = True,
    exist_ok = True,
)


## Baseline Training Results

### Load Training Results from CSV

In [ ]:
'''
load the result CSV file locally in the VM
'''
results_csv = RUNS_DIR / BASELINE_NAME / 'results.csv'

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

print('Columns:', df.columns.tolist())
df.head()

In [ ]:
'''
map col names to something more readable for later data analysis
'''
def find_col(df, patterns):
    # return the first col name that matches
    # any of the given substrs
    for p in patterns:
        matches = [c for c in df.columns if p.lower() in c.lower()]
        if matches:
            return matches[0]
    return None

epoch_col      = find_col(df, ['epoch'])
train_box_col  = find_col(df, ['train/box_loss', 'train/box'])
train_cls_col  = find_col(df, ['train/cls_loss', 'train/cls'])
train_dfl_col  = find_col(df, ['train/dfl_loss', 'train/dfl'])
val_box_col    = find_col(df, ['val/box_loss',   'val/box'])
val_cls_col    = find_col(df, ['val/cls_loss',   'val/cls'])
val_dfl_col    = find_col(df, ['val/dfl_loss',   'val/dfl'])
map50_col      = find_col(df, ['metrics/mAP50(B)', 'metrics/mAP50'])
map50_95_col   = find_col(df, ['metrics/mAP50-95(B)', 'metrics/mAP50-95'])
precision_col  = find_col(df, ['metrics/precision(B)', 'metrics/precision'])
recall_col     = find_col(df, ['metrics/recall(B)', 'metrics/recall'])

epochs = df[epoch_col].values

# composite training loss & validation loss (sum of box + cls + dfl)
df['train_total_loss'] = df[[c for c in [train_box_col, train_cls_col, train_dfl_col] if c]].sum(axis=1)
df['val_total_loss']   = df[[c for c in [val_box_col,   val_cls_col,   val_dfl_col]   if c]].sum(axis=1)

print('Columns found:')
for name, col in [
    ('epoch', epoch_col), ('train box', train_box_col), ('train cls', train_cls_col),
    ('train dfl', train_dfl_col), ('val box', val_box_col), ('val cls', val_cls_col),
    ('val dfl', val_dfl_col), ('mAP50', map50_col), ('mAP50-95', map50_95_col),
    ('precision', precision_col), ('recall', recall_col)
]:
    print(f'  {name:12s} -> {col}')

### Determine the Best Epoch According to `mAP@0.5`

In [ ]:

'''
print a summary of the best epoch according to mAP@0.5
'''
if map50_col:
    best_idx = df[map50_col].idxmax()
    best_row = df.loc[best_idx]
    best_ep  = int(best_row[epoch_col])

    print('╔' + '═' * 37 + '╗')
    print(f'║  BEST EPOCH by mAP@0.5: Epoch {best_ep:<4}  ║')
    print('╠' + '═' * 37 + '╣')

    metrics_to_show = [
        ('mAP@0.5',      map50_col),
        ('mAP@0.5:0.95', map50_95_col),
        ('Precision',    precision_col),
        ('Recall',       recall_col),
        ('Train Loss',   'train_total_loss'),
        ('Val Loss',     'val_total_loss'),
    ]
    for label, col in metrics_to_show:
        if col and col in df.columns:
            print(f'║  {label:<18}: {best_row[col]:<15.4f}║')

    print('╚' + '═' * 37 + '╝')

# Final epoch summary table
summary_cols = [epoch_col]
for c in [train_box_col, train_cls_col, train_dfl_col,
          val_box_col, val_cls_col, val_dfl_col,
          map50_col, map50_95_col, precision_col, recall_col]:
    if c:
        summary_cols.append(c)

print('\nLast 5 epochs:')
df[summary_cols].tail(5).style.format('{:.4f}').set_caption('Training Results')

### Loss Curve and Fitting Analysis



#### Individual Loss Component Curves

YOLOv11 splits loss into three components:

1. Bounding Box Regression Loss (Box Loss)
2. Classification Loss
3. Distribution Focal Loss (DFL)

In [ ]:
'''
Makes three graphs side-by-side: One for each individually loss
component's training and validation curves
'''
plt.style.use('seaborn-v0_8-whitegrid')

loss_pairs = [
    (train_box_col, val_box_col, 'Box Loss',  'royalblue', 'darkorange'),
    (train_cls_col, val_cls_col, 'Classification Loss','royalblue', 'darkorange'),
    (train_dfl_col, val_dfl_col, 'Distribution Focal Loss (DFL)',  'royalblue', 'darkorange'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), dpi=FIG_DPI)
fig.suptitle('YOLOv11n Baseline Training: Individual Loss Components', fontsize=14, fontweight='bold', y=1.02)

for ax, (t_col, v_col, title, tc, vc) in zip(axes, loss_pairs):
    if t_col:
        ax.plot(epochs, df[t_col], color=tc, linewidth=1.8, label='Train')
    if v_col:
        ax.plot(epochs, df[v_col], color=vc, linewidth=1.8, linestyle='--', label='Val')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=10)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
fig.savefig(RUNS_DIR / BASELINE_NAME / 'loss_components.png', bbox_inches='tight')
plt.show()
print('Saved loss_components.png')

#### Composite Loss Curve (Train Loss vs. Validation Loss)

In [ ]:
'''
Plot the training loss and validation loss curves.
Also shows the best epoch according to mAP@0.5
'''
fig, ax = plt.subplots(figsize=(10, 5), dpi=FIG_DPI)

ax.plot(epochs, df['train_total_loss'], color='royalblue', linewidth=2.2,
        label='Training Loss', marker='o', markersize=3)
ax.plot(epochs, df['val_total_loss'],   color='darkorange',    linewidth=2.2,
        label='Validation Loss', linestyle='--', marker='s', markersize=3)

# Annotate the best validation epoch
if map50_col:
    best_ep = int(best_row[epoch_col])
    best_val_loss = df.loc[best_idx, 'val_total_loss']
    ax.axvline(best_ep, color='#E2E535', linestyle=':', linewidth=1.5, alpha=0.7, label=f'Best mAP@0.5 (epoch {best_ep})')

ax.set_title('YOLOv11n Baseline: Composite Training Loss vs. Validation Loss', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Total Loss (box + cls + dfl)', fontsize=11)
ax.legend(fontsize=9)
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
fig.savefig(RUNS_DIR / BASELINE_NAME / 'composite_loss_curve.png', bbox_inches='tight')
plt.show()
print('Saved composite_loss_curve.png')

#### mAP, Precision, and Recall Over the Training Period



In [ ]:
'''
Makes two graphs side-by-side: one for plotting the mAP curves,
and one for plotting the precision & recall curves
'''
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=FIG_DPI)
fig.suptitle('YOLOv11n Baseline: Detection Metrics Over Training Period', fontsize=13, fontweight='bold')

# graph the mAP over the training period
ax = axes[0]
if map50_col:
    ax.plot(epochs, df[map50_col], color='royalblue',  linewidth=2, label='mAP@0.5')
if map50_95_col:
    ax.plot(epochs, df[map50_95_col], color='darkorange', linewidth=2, label='mAP@0.5:0.95')

ax.set_title('Mean Average Precision (mAP)')
ax.set_xlabel('Epoch'); ax.set_ylabel('mAP')
ax.legend(); ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# graph precision and recall
ax = axes[1]
if precision_col:
    ax.plot(epochs, df[precision_col], color='mediumseagreen', linewidth=2, label='Precision')
if recall_col:
    ax.plot(epochs, df[recall_col],    color='mediumvioletred', linewidth=2, label='Recall')

ax.set_title('Precision & Recall')
ax.set_xlabel('Epoch'); ax.set_ylabel('Score')
ax.legend(); ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
fig.savefig(RUNS_DIR / BASELINE_NAME / 'detection_metrics.png', bbox_inches='tight')
plt.show()
print('Saved detection_metrics.png')

#### Overfitting / Underfitting Diagnostic

In [ ]:
'''
Makes two graphs side-by-side: one for plotting the difference between
validation loss & training loss for each epoch, and one for plotting
the smoothed rolling mean of validation loss to show trend
'''
df['loss_gap'] = df['val_total_loss'] - df['train_total_loss']

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=FIG_DPI)
fig.suptitle('YOLOv11n Baseline: Overfitting/Underfitting Diagnostics', fontsize=13, fontweight='bold')

# graph the difference between validation loss and training loss
ax = axes[0]
ax.bar(epochs, df['loss_gap'],
       color=['darkorange' if g > 0 else 'royalblue' for g in df['loss_gap']],
       alpha=0.8, width=0.7)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Difference between Validation Loss & Training Loss per Epoch')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss Gap')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# graph the smoothed rolling mean of validation loss to show trend
ax = axes[1]
window = 5
val_smooth = df['val_total_loss'].rolling(window, min_periods=1).mean()
train_smooth = df['train_total_loss'].rolling(window, min_periods=1).mean()

ax.plot(epochs, df['train_total_loss'], color='royalblue', alpha=0.3, linewidth=1)
ax.plot(epochs, train_smooth, color='royalblue', linewidth=2.2, label=f'Train (smooth {window})')
ax.plot(epochs, df['val_total_loss'], color='darkorange', alpha=0.3, linewidth=1)
ax.plot(epochs, val_smooth, color='darkorange', linewidth=2.2, linestyle='--', label=f'Val (smooth {window})')
ax.set_title('Smoothed Loss Curves')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
fig.savefig(RUNS_DIR / BASELINE_NAME / 'fitting_diagnostics.png', bbox_inches='tight')
plt.show()
print('Saved fitting_diagnostics.png')

#### Quantitative Fitting Summary

In [ ]:
'''
print a quantitative summary of the model's training
'''
early_window  = epochs[:10] if len(epochs) >= 10 else epochs
late_window   = epochs[-10:] if len(epochs) >= 10 else epochs

early_gap = df['loss_gap'].iloc[:len(early_window)].mean()
late_gap  = df['loss_gap'].iloc[-len(late_window):].mean()
max_gap   = df['loss_gap'].max()
gap_trend = late_gap - early_gap # positive = gap widening over time

final_train_loss = df['train_total_loss'].iloc[-1]
final_val_loss   = df['val_total_loss'].iloc[-1]
final_gap_pct    = (final_val_loss - final_train_loss) / final_train_loss * 100

final_map50    = df[map50_col].iloc[-1]    if map50_col    else None
final_map5095  = df[map50_95_col].iloc[-1] if map50_95_col else None
best_map50     = df[map50_col].max()       if map50_col    else None

print('╔══════════════════════════════════════════════════════╗')
print('║     QUANTITATIVE FITTING SUMMARY for YOLOv11n        ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Epochs trained             :{len(epochs):<24}║')
print(f'║  Final train loss           :{final_train_loss:<24.4f}║')
print(f'║  Final val loss             :{final_val_loss:<24.4f}║')
print(f'║  Val−Train gap (final)      :{final_val_loss - final_train_loss:<24.4f}║')
print(f'║  Val−Train gap (final, %)   :{final_gap_pct:<24.2f}║')
print(f'║  Mean gap (early 10 epochs) :{early_gap:<24.4f}║')
print(f'║  Mean gap (late  10 epochs) :{late_gap:<24.4f}║')
print(f'║  Gap trend (late − early)   :{gap_trend:<+24.4f}║')
print(f'║  Max gap observed           :{max_gap:<24.4f}║')
if best_map50 is not None:
    print(f'║  Best mAP@0.5               :{best_map50:<24.4f}║')
if final_map50 is not None:
    print(f'║  Final mAP@0.5              :{final_map50:<24.4f}║')
if final_map5095 is not None:
    print(f'║  Final mAP@0.5:0.95         :{final_map5095:<24.4f}║')
print('╚══════════════════════════════════════════════════════╝')

#### Automated Fitting Diagnosis

In [ ]:
'''
determine if there is evidence of convergence and/or overfitting/underfitting
'''
import textwrap

W   = 54 # inner width
SEP = '╠' + '═' * W + '╣'

def _row(text=''):
    return f'║  {text:<{W - 2}}║'

def _wrap(text, prefix=''):
    available = W - 2 - len(prefix)
    lines = textwrap.wrap(text, available)
    return [_row(prefix + ln) for ln in lines] if lines else [_row()]

OVERFIT_THRESHOLD   = 0.15   # overfitting present if validation > train by more than 15%
UNDERFIT_MAP_THRESH = 0.20   # underfitting present if mAP50 < 20%
CONVERGE_STD_THRESH = 0.005  # convergence present if std of last 10 val losses < 0.005

last_10_val_std = df['val_total_loss'].iloc[-10:].std()
is_converged    = last_10_val_std < CONVERGE_STD_THRESH
is_overfit      = (final_gap_pct > OVERFIT_THRESHOLD * 100)
is_underfit     = (best_map50 < UNDERFIT_MAP_THRESH) if best_map50 is not None else False

print('╔' + '═' * W + '╗')
print(_row('FITTING DIAGNOSIS for YOLOv11n Baseline'))
print(SEP)

if is_converged:
    print(_row('> CONVERGED'))
    for ln in _wrap('Validation loss stabilized in the last 10 epochs.', '  '):
        print(ln)
else:
    print(_row('> NOT YET CONVERGED'))
    for ln in _wrap('Validation loss still decreasing; more epochs may help.', '  '):
        print(ln)

print(_row())
print(SEP)

if is_overfit:
    print(_row('> OVERFITTING DETECTED'))
    for ln in _wrap(f'Val loss exceeds train loss by {final_gap_pct:.1f}%.', '  '):
        print(ln)
    for ln in _wrap('Mitigations: more data / stronger augmentation / weight decay / dropout.', '  '):
        print(ln)
elif is_underfit:
    print(_row('> UNDERFITTING DETECTED'))
    for ln in _wrap(f'Best mAP@0.5 only {best_map50:.3f}.', '  '):
        print(ln)
    for ln in _wrap('Mitigations: larger model (yolo11s / m), more epochs, higher resolution.', '  '):
        print(ln)
else:
    print(_row('> GOOD FIT'))
    for ln in _wrap('Val and train losses track closely; no severe over/underfitting.', '  '):
        print(ln)

print(_row())
print(SEP)
print(_row(f'Last-10 val loss std : {last_10_val_std:.5f}  (threshold: {CONVERGE_STD_THRESH})'))
print(_row(f'Convergence          : {"Yes" if is_converged else "No"}'))
print(_row(f'Overfit signal       : {"Yes" if is_overfit   else "No"}'))
print(_row(f'Underfit signal      : {"Yes" if is_underfit  else "No"}'))
print('╚' + '═' * W + '╝')

### Baseline Summary Table

In [ ]:
'''
prints an overall summary of the baseline training
'''
import warnings
warnings.filterwarnings('ignore')

summary = {
    'Model'        : 'YOLOv11n (nano)',
    'Dataset'      : 'VisDrone2019-DET-train',
    'Train images' : n_train,
    'Val images'   : n_val,
    'Epochs'       : int(epochs[-1]),
    'Image size'   : 640,
    'Batch size'   : 16,
    'Final Train Loss'   : f'{final_train_loss:.4f}',
    'Final Val Loss'     : f'{final_val_loss:.4f}',
    'Val−Train Gap (%)'  : f'{final_gap_pct:.2f}',
    'Best mAP@0.5'       : f'{best_map50:.4f}' if best_map50 else 'N/A',
    'mAP@0.5:0.95'       : f'{final_map5095:.4f}' if final_map5095 else 'N/A',
    'Convergence'        : 'Yes' if is_converged else 'No',
    'Overfit Signal'     : 'Yes' if is_overfit  else 'No',
    'Underfit Signal'    : 'Yes' if is_underfit else 'No',
}

summary_df = pd.DataFrame.from_dict(summary, orient='index', columns=['Value'])
summary_df.index.name = 'Metric'
print(summary_df.to_string())

summary_df.to_csv(RUNS_DIR / BASELINE_NAME / 'baseline_summary.csv')
print('\nSaved baseline_summary.csv')

## Controlled Experiment

For this experiment, image resolution is doubled to `1280 px`. To handle larger resolutions, batch size will be reduced by 75% to 16/32 (depending on the GPU's RAM). A reduced batch size may independently affect generalization, but it necessary due to the increase in VRAM needed to train the model.

### Train the Experiment Model

In [ ]:
import torch
from ultralytics import YOLO

model = YOLO('yolo11n.pt') # Load pretrained weights

gpu  = torch.cuda.get_device_properties(0)
vram = gpu.total_memory / 1e9
print(f'GPU  : {gpu.name}')
print(f'VRAM : {vram:.1f} GB')
print(f'CUDA : {torch.version.cuda}')

BATCH_SIZE = 16 if vram < 70 else 32
print(f'Batch: {BATCH_SIZE}  (auto-selected for {vram:.0f} GB VRAM)')

model = YOLO('yolo11n.pt') # Load pretrained weights

results = model.train(
    data     = str(yaml_path),
    epochs   = 50,
    imgsz    = 1280, # new image res
    batch    = BATCH_SIZE,
    workers  = 8, # parallel dataloaders
    cache    = 'ram', # load all images into RAM once at the start
    amp      = True, # Automatic Mixed Precision
    cos_lr   = True, # cosine LR decay
    name     = EXP_NAME,
    project  = str(RUNS_DIR),
    seed     = SEED,
    verbose  = True,
    exist_ok = True,
)


## Experiment Model Results

### Load Baseline and Experiment Models' Results from CSVs

In [ ]:
# Load baseline results
df_b = pd.read_csv(RUNS_DIR / BASELINE_NAME / 'results.csv')
df_b.columns = df_b.columns.str.strip()
df_b['train_total_loss'] = df_b[[c for c in [train_box_col, train_cls_col, train_dfl_col] if c]].sum(axis=1)
df_b['val_total_loss']   = df_b[[c for c in [val_box_col,   val_cls_col,   val_dfl_col]   if c]].sum(axis=1)

# Load experimental results
df_e = pd.read_csv(RUNS_DIR / EXP_NAME / 'results.csv')
df_e.columns = df_e.columns.str.strip()
df_e['train_total_loss'] = df_e[[c for c in [train_box_col, train_cls_col, train_dfl_col] if c]].sum(axis=1)
df_e['val_total_loss']   = df_e[[c for c in [val_box_col,   val_cls_col,   val_dfl_col]   if c]].sum(axis=1)

epochs_b = df_b[epoch_col].values
epochs_e = df_e[epoch_col].values

print(f'Baseline epochs  : {len(epochs_b)}')
print(f'Experiment epochs: {len(epochs_e)}')

### Experiment Data Analysis, Visualization, and Comparison

In [ ]:
'''
Makes three graphs side-by-side: one for plotting the mAP@0.50 curves,
one for plotting the mAP@0.50:0.95 curves, and one for plotting the
validation loss curves
'''
fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=FIG_DPI)
fig.suptitle('Baseline vs. Experiment: mAP and Validation Loss', fontsize=13, fontweight='bold')

# mAP@0.5
ax = axes[0]
if map50_col:
    ax.plot(epochs_b, df_b[map50_col], color='royalblue',  linewidth=2, label='Baseline (640px)')
    ax.plot(epochs_e, df_e[map50_col], color='darkorange', linewidth=2, linestyle='--', label='Exp (1280px)')
ax.set_title('mAP@0.5')
ax.set_xlabel('Epoch'); ax.set_ylabel('mAP')
ax.legend(); ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# mAP@0.5:0.95
ax = axes[1]
if map50_95_col:
    ax.plot(epochs_b, df_b[map50_95_col], color='royalblue',  linewidth=2, label='Baseline (640px)')
    ax.plot(epochs_e, df_e[map50_95_col], color='darkorange', linewidth=2, linestyle='--', label='Exp (1280px)')
ax.set_title('mAP@0.5:0.95')
ax.set_xlabel('Epoch'); ax.set_ylabel('mAP')
ax.legend(); ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# Validation loss
ax = axes[2]
ax.plot(epochs_b, df_b['val_total_loss'], color='royalblue',  linewidth=2, label='Baseline (640px)')
ax.plot(epochs_e, df_e['val_total_loss'], color='darkorange', linewidth=2, linestyle='--', label='Exp (1280px)')
ax.set_title('Validation Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Total Loss')
ax.legend(); ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
fig.savefig(RUNS_DIR / EXP_NAME / 'comparison_metrics.png', bbox_inches='tight')
plt.show()
print('Saved comparison_metrics.png')

In [ ]:
'''
makes two graphs side-by-side: one for plotting the baseline's composite
training and validation loss curves, and one for plotting experiment's
composite training and validation loss curves

Both graphs show the best epoch according to mAP@0.5
'''
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=FIG_DPI)
fig.suptitle('Baseline vs Experiment: Composite Training & Validation Loss', fontsize=13, fontweight='bold')

for ax, df_, epochs_, title in [
    (axes[0], df_b, epochs_b, 'Baseline (640px)'),
    (axes[1], df_e, epochs_e, 'Experiment (1280px)'),
]:
    ax.plot(epochs_, df_['train_total_loss'], color='royalblue',  linewidth=2, label='Train')
    ax.plot(epochs_, df_['val_total_loss'],   color='darkorange', linewidth=2, linestyle='--', label='Val')

    # Annotate the best mAP50 epoch for this model
    if map50_col and map50_col in df_.columns:
        best_idx_  = df_[map50_col].idxmax()
        best_ep_   = int(df_.loc[best_idx_, epoch_col])
        best_loss_ = df_.loc[best_idx_, 'val_total_loss']
        ax.axvline(best_ep_, color='#E2E535', linestyle=':', linewidth=1.5, alpha=0.9,
                   label=f'Best mAP@0.5 (epoch {best_ep_})')

    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Total Loss')
    ax.legend(); ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
fig.savefig(RUNS_DIR / EXP_NAME / 'comparison_loss_curves.png', bbox_inches='tight')
plt.show()
print('Saved comparison_loss_curves.png')

In [ ]:
'''
Print a quantitative comparison summary of the two models
'''
def run_stats(df_):
    best_idx_ = df_[map50_col].idxmax() if map50_col else 0
    best_row_ = df_.loc[best_idx_]
    final_train_ = df_['train_total_loss'].iloc[-1]
    final_val_   = df_['val_total_loss'].iloc[-1]
    gap_pct_     = (final_val_ - final_train_) / final_train_ * 100
    late_std_    = df_['val_total_loss'].iloc[-10:].std()
    return {
        'Best mAP@0.5'     : best_row_[map50_col]     if map50_col     else float('nan'),
        'Best mAP@0.5:0.95': best_row_[map50_95_col]  if map50_95_col  else float('nan'),
        'Best Precision'   : best_row_[precision_col]  if precision_col else float('nan'),
        'Best Recall'      : best_row_[recall_col]     if recall_col    else float('nan'),
        'Final Train Loss' : final_train_,
        'Final Val Loss'   : final_val_,
        'Val-Train Gap (%)': gap_pct_,
        'Late-10 Val Std'  : late_std_,
    }

stats_b = run_stats(df_b)
stats_e = run_stats(df_e)

W2 = 54
print('╔' + '═' * W2 + '╗')
print(f'║  {"QUANTITATIVE COMPARISON":<{W2 - 2}}║')
print('╠' + '═' * W2 + '╣')
print(f'║  {"Metric":<22}{"Baseline (640px)":<18}{"Exp (1280px)":<12}║')
print('╠' + '═' * W2 + '╣')
for metric in stats_b:
    b_val = stats_b[metric]
    e_val = stats_e[metric]
    delta = e_val - b_val
    arrow = '▲' if delta > 0 else '▼'
    print(f'║  {metric:<22}{b_val:<18.4f}{e_val:<12.4f}║')
print('╚' + '═' * W2 + '╝')

## Iterative Improvement Cycles

### Improvement Cycle 1: Increase the number of epochs to 65

In [ ]:
import torch
from ultralytics import YOLO

model = YOLO('yolo11n.pt') # Load pretrained weights

gpu  = torch.cuda.get_device_properties(0)
vram = gpu.total_memory / 1e9
print(f'GPU  : {gpu.name}')
print(f'VRAM : {vram:.1f} GB')
print(f'CUDA : {torch.version.cuda}')

BATCH_SIZE = 16 if vram < 70 else 32
print(f'Batch: {BATCH_SIZE}  (auto-selected for {vram:.0f} GB VRAM)')

results = model.train(
    data      = str(yaml_path),
    epochs    = 65, # 65 epochs instead of 50
    imgsz     = 1280,
    batch     = BATCH_SIZE,
    workers   = 8,     # parallel dataloaders
    cache     = 'ram', # load all images into RAM once at the start
    amp       = True,  # Automatic Mixed Precision
    cos_lr    = True,  # cosine LR decay
    name      = IMP_NAME1,
    project   = str(RUNS_DIR),
    seed      = SEED,
    verbose   = True,
    exist_ok  = True,
)


### Load the Experiment and Improvement Models' Results from CSVs

In [ ]:

# Load the experiment's results
df_e = pd.read_csv(RUNS_DIR / EXP_NAME / 'results.csv')
df_e.columns = df_e.columns.str.strip()

epoch_col      = find_col(df_e, ['epoch'])
train_box_col  = find_col(df_e, ['train/box_loss', 'train/box'])
train_cls_col  = find_col(df_e, ['train/cls_loss', 'train/cls'])
train_dfl_col  = find_col(df_e, ['train/dfl_loss', 'train/dfl'])
val_box_col    = find_col(df_e, ['val/box_loss',   'val/box'])
val_cls_col    = find_col(df_e, ['val/cls_loss',   'val/cls'])
val_dfl_col    = find_col(df_e, ['val/dfl_loss',   'val/dfl'])
map50_col      = find_col(df_e, ['metrics/mAP50(B)', 'metrics/mAP50'])
map50_95_col   = find_col(df_e, ['metrics/mAP50-95(B)', 'metrics/mAP50-95'])
precision_col  = find_col(df_e, ['metrics/precision(B)', 'metrics/precision'])
recall_col     = find_col(df_e, ['metrics/recall(B)', 'metrics/recall'])

epochs = df_e[epoch_col].values


df_e['train_total_loss'] = df_e[[c for c in [train_box_col, train_cls_col, train_dfl_col] if c]].sum(axis=1)
df_e['val_total_loss']   = df_e[[c for c in [val_box_col,   val_cls_col,   val_dfl_col]   if c]].sum(axis=1)

# Load the improvement's results
df_imp = pd.read_csv(RUNS_DIR / IMP_NAME1 / 'results.csv')
df_imp.columns = df_imp.columns.str.strip()

epoch_col      = find_col(df_imp, ['epoch'])
train_box_col  = find_col(df_imp, ['train/box_loss', 'train/box'])
train_cls_col  = find_col(df_imp, ['train/cls_loss', 'train/cls'])
train_dfl_col  = find_col(df_imp, ['train/dfl_loss', 'train/dfl'])
val_box_col    = find_col(df_imp, ['val/box_loss',   'val/box'])
val_cls_col    = find_col(df_imp, ['val/cls_loss',   'val/cls'])
val_dfl_col    = find_col(df_imp, ['val/dfl_loss',   'val/dfl'])
map50_col      = find_col(df_imp, ['metrics/mAP50(B)', 'metrics/mAP50'])
map50_95_col   = find_col(df_imp, ['metrics/mAP50-95(B)', 'metrics/mAP50-95'])
precision_col  = find_col(df_imp, ['metrics/precision(B)', 'metrics/precision'])
recall_col     = find_col(df_imp, ['metrics/recall(B)', 'metrics/recall'])

epochs = df_imp[epoch_col].values

df_imp['train_total_loss'] = df_imp[[c for c in [train_box_col, train_cls_col, train_dfl_col] if c]].sum(axis=1)
df_imp['val_total_loss']   = df_imp[[c for c in [val_box_col,   val_cls_col,   val_dfl_col]   if c]].sum(axis=1)

epochs_e = df_e[epoch_col].values
epochs_imp = df_imp[epoch_col].values

print(f'Experiment epochs : {len(epochs_e)}')
print(f'Improvement epochs: {len(epochs_imp)}')

## Comparative Analysis: Experiment Model vs. Improvement Cycle 1 Model


In [ ]:
'''
Plot the results for the improvement model
'''
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, axes = plt.subplots(2, 3, figsize=(18, 10), dpi=FIG_DPI)
fig.suptitle(f'Improvement Cycle 1: Training & Validation Metrics', fontsize=16, fontweight='bold')

# box loss plot
axes[0, 0].plot(epochs_imp, df_imp['train/box_loss'], label='Train', color='royalblue', linewidth=2)
axes[0, 0].plot(epochs_imp, df_imp['val/box_loss'], label='Val', color='darkorange', linestyle='--', linewidth=2)
axes[0, 0].set_title('Box Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()

# class loss plot
axes[0, 1].plot(epochs_imp, df_imp['train/cls_loss'], label='Train', color='royalblue', linewidth=2)
axes[0, 1].plot(epochs_imp, df_imp['val/cls_loss'], label='Val', color='darkorange', linestyle='--', linewidth=2)
axes[0, 1].set_title('Class Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()

# DFL plot
axes[0, 2].plot(epochs_imp, df_imp['train/dfl_loss'], label='Train', color='royalblue', linewidth=2)
axes[0, 2].plot(epochs_imp, df_imp['val/dfl_loss'], label='Val', color='darkorange', linestyle='--', linewidth=2)
axes[0, 2].set_title('Distribution Focal Loss (DFL)')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Loss')
axes[0, 2].legend()

# mAP@0.50 plot
axes[1, 0].plot(epochs_imp, df_imp['metrics/mAP50(B)'], color='royalblue', linewidth=2)
axes[1, 0].set_title('mAP@0.50')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('mAP')

# mAP@0.50:0.95 plot
axes[1, 1].plot(epochs_imp, df_imp['metrics/mAP50-95(B)'], color='darkorange', linewidth=2)
axes[1, 1].set_title('mAP@0.50:0.95')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('mAP')

# precision & recall plot
axes[1, 2].plot(epochs_imp, df_imp['metrics/precision(B)'], label='Precision', color='mediumseagreen', linewidth=2)
axes[1, 2].plot(epochs_imp, df_imp['metrics/recall(B)'], label='Recall', color='dodgerblue', linewidth=2)
axes[1, 2].set_title('Precision & Recall')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('Score')
axes[1, 2].legend()

for ax in axes.flat:
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

fig.savefig(RUNS_DIR / IMP_NAME1 / 'comparison_loss_curves.png', bbox_inches='tight')
plt.show()
print('Saved comparison_loss_curves.png')

In [ ]:
'''
Plot the composite loss for the experimental and
improvement models side-by-side for comparison
'''

# calculate composite loss for the improvement model
df_imp['train_total_loss'] = df_imp[[c for c in [train_box_col, train_cls_col, train_dfl_col] if c]].sum(axis=1)
df_imp['val_total_loss']   = df_imp[[c for c in [val_box_col,   val_cls_col,   val_dfl_col]   if c]].sum(axis=1)
epochs_imp = df_imp[epoch_col].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=FIG_DPI)
fig.suptitle('Experiment Model vs Improvement Cycle 1: Composite Training & Validation Loss', fontsize=13, fontweight='bold')

for ax, df_, epochs_, title in [
    (axes[0], df_e, epochs_e, 'Experiment Model'),
    (axes[1], df_imp, epochs_imp, 'Improvement Model'),
]:
    ax.plot(epochs_, df_['train_total_loss'], color='royalblue',  linewidth=2, label='Train')
    ax.plot(epochs_, df_['val_total_loss'],   color='darkorange', linewidth=2, linestyle='--', label='Val')

    # Annotate the best mAP@0.50 epoch for this model
    if map50_col and map50_col in df_.columns:
        best_idx_  = df_[map50_col].idxmax()
        best_ep_   = int(df_.loc[best_idx_, epoch_col])
        ax.axvline(best_ep_, color='#E2E535', linestyle=':', linewidth=1.5, alpha=0.9,
                   label=f'Best mAP@0.5 (epoch {best_ep_})')

    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Total Loss')
    ax.legend(); ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
fig.savefig(RUNS_DIR / IMP_NAME1 / 'exp_vs_imp_loss_curves.png', bbox_inches='tight')
plt.show()
print('Saved exp_vs_imp_loss_curves.png')

In [ ]:
'''
Plot mAP@0.50, mAP@0.50:0.95, and validation loss for
the experimental and improvement models side-by-side
for comparison.
'''
fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=FIG_DPI)
fig.suptitle('Experiment vs. Improvement Cycle 1: mAP and Validation Loss', fontsize=13, fontweight='bold')

# draw the mAP@0.5 curves
ax = axes[0]
if map50_col:
    ax.plot(epochs_e, df_e[map50_col], color='royalblue',  linewidth=2, label='Experimental')
    ax.plot(epochs_imp, df_imp[map50_col], color='darkorange', linewidth=2, linestyle='--', label='Improvement')
ax.set_title('mAP@0.5')
ax.set_xlabel('Epoch'); ax.set_ylabel('mAP')
ax.legend(); ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# draw the mAP@0.5:0.95 curves
ax = axes[1]
if map50_95_col:
    ax.plot(epochs_e, df_e[map50_95_col], color='royalblue',  linewidth=2, label='Experimental')
    ax.plot(epochs_imp, df_imp[map50_95_col], color='darkorange', linewidth=2, linestyle='--', label='Improvement')
ax.set_title('mAP@0.5:0.95')
ax.set_xlabel('Epoch'); ax.set_ylabel('mAP')
ax.legend(); ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# draw the validation loss curves
ax = axes[2]
ax.plot(epochs_e, df_e['val_total_loss'], color='royalblue',  linewidth=2, label='Experimental')
ax.plot(epochs_imp, df_imp['val_total_loss'], color='darkorange', linewidth=2, linestyle='--', label='Improvement')
ax.set_title('Validation Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Total Loss')
ax.legend(); ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
fig.savefig(RUNS_DIR / IMP_NAME1 / 'exp_vs_imp_metrics.png', bbox_inches='tight')
plt.show()
print('Saved exp_vs_imp_metrics.png')

## Improvement Cycle 2: Use SGD as the Optimization Algorithm

For the baseline, experiment, and improvement cycle 1 models, `optimizer='auto'` was used, so Ultralytics selected AdamW optimizer. This improvement cycle will build off of the first improvement cycle, and will use SGD instead of AdamW.

### Training Cycle 2

In [ ]:
import torch
from ultralytics import YOLO

model = YOLO('yolo11n.pt') # Load pretrained weights

gpu  = torch.cuda.get_device_properties(0)
vram = gpu.total_memory / 1e9
print(f'GPU  : {gpu.name}')
print(f'VRAM : {vram:.1f} GB')
print(f'CUDA : {torch.version.cuda}')

BATCH_SIZE = 16 if vram < 70 else 32
print(f'Batch: {BATCH_SIZE}  (auto-selected for {vram:.0f} GB VRAM)')

results = model.train(
    data      = str(yaml_path),
    epochs    = 65, # 65 epochs instead of 50 (from cycle 1)
    imgsz     = 1280,
    batch     = BATCH_SIZE,
    workers   = 8,     # parallel dataloaders
    cache     = 'ram', # load all images into RAM once at the start
    amp       = True,  # Automatic Mixed Precision
    cos_lr    = True,  # cosine LR decay
    optimizer = 'SGD', # use SGD instead of AdamW
    name      = IMP_NAME2,
    project   = str(RUNS_DIR),
    seed      = SEED,
    verbose   = True,
    exist_ok  = True,
)


### Load the Improvement Cycles' Results from CSVs

In [ ]:

# Load the first improvement cycle's results
df_imp = pd.read_csv(RUNS_DIR / IMP_NAME1 / 'results.csv')
df_imp.columns = df_imp.columns.str.strip()

epoch_col      = find_col(df_imp, ['epoch'])
train_box_col  = find_col(df_imp, ['train/box_loss', 'train/box'])
train_cls_col  = find_col(df_imp, ['train/cls_loss', 'train/cls'])
train_dfl_col  = find_col(df_imp, ['train/dfl_loss', 'train/dfl'])
val_box_col    = find_col(df_imp, ['val/box_loss',   'val/box'])
val_cls_col    = find_col(df_imp, ['val/cls_loss',   'val/cls'])
val_dfl_col    = find_col(df_imp, ['val/dfl_loss',   'val/dfl'])
map50_col      = find_col(df_imp, ['metrics/mAP50(B)', 'metrics/mAP50'])
map50_95_col   = find_col(df_imp, ['metrics/mAP50-95(B)', 'metrics/mAP50-95'])
precision_col  = find_col(df_imp, ['metrics/precision(B)', 'metrics/precision'])
recall_col     = find_col(df_imp, ['metrics/recall(B)', 'metrics/recall'])

epochs = df_imp[epoch_col].values

df_imp['train_total_loss'] = df_imp[[c for c in [train_box_col, train_cls_col, train_dfl_col] if c]].sum(axis=1)
df_imp['val_total_loss']   = df_imp[[c for c in [val_box_col,   val_cls_col,   val_dfl_col]   if c]].sum(axis=1)

# Load the second improvement cycle's results
df_imp2 = pd.read_csv(RUNS_DIR / IMP_NAME2 / 'results.csv')
df_imp2.columns = df_imp2.columns.str.strip()

epoch_col      = find_col(df_imp2, ['epoch'])
train_box_col  = find_col(df_imp2, ['train/box_loss', 'train/box'])
train_cls_col  = find_col(df_imp2, ['train/cls_loss', 'train/cls'])
train_dfl_col  = find_col(df_imp2, ['train/dfl_loss', 'train/dfl'])
val_box_col    = find_col(df_imp2, ['val/box_loss',   'val/box'])
val_cls_col    = find_col(df_imp2, ['val/cls_loss',   'val/cls'])
val_dfl_col    = find_col(df_imp2, ['val/dfl_loss',   'val/dfl'])
map50_col      = find_col(df_imp2, ['metrics/mAP50(B)', 'metrics/mAP50'])
map50_95_col   = find_col(df_imp2, ['metrics/mAP50-95(B)', 'metrics/mAP50-95'])
precision_col  = find_col(df_imp2, ['metrics/precision(B)', 'metrics/precision'])
recall_col     = find_col(df_imp2, ['metrics/recall(B)', 'metrics/recall'])

epochs = df_imp2[epoch_col].values

df_imp2['train_total_loss'] = df_imp2[[c for c in [train_box_col, train_cls_col, train_dfl_col] if c]].sum(axis=1)
df_imp2['val_total_loss']   = df_imp2[[c for c in [val_box_col,   val_cls_col,   val_dfl_col]   if c]].sum(axis=1)

epochs_imp = df_imp[epoch_col].values
epochs_imp2 = df_imp2[epoch_col].values

print(f'Experiment epochs : {len(epochs_e)}')
print(f'Improvement epochs: {len(epochs_imp)}')

In [ ]:
'''
Plots the loss curves for the improvement cycle model
'''
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.style.use('seaborn-v0_8-whitegrid')

fig, axes = plt.subplots(2, 3, figsize=(18, 10), dpi=FIG_DPI)
fig.suptitle(f'Improvement Cycle 2: Training & Validation Metrics', fontsize=16, fontweight='bold')

# box loss plot
axes[0, 0].plot(epochs_imp2, df_imp2['train/box_loss'], label='Train', color='royalblue', linewidth=2)
axes[0, 0].plot(epochs_imp2, df_imp2['val/box_loss'], label='Val', color='darkorange', linestyle='--', linewidth=2)
axes[0, 0].set_title('Box Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()

# class loss plot
axes[0, 1].plot(epochs_imp2, df_imp2['train/cls_loss'], label='Train', color='royalblue', linewidth=2)
axes[0, 1].plot(epochs_imp2, df_imp2['val/cls_loss'], label='Val', color='darkorange', linestyle='--', linewidth=2)
axes[0, 1].set_title('Class Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()

# DFL plot
axes[0, 2].plot(epochs_imp2, df_imp2['train/dfl_loss'], label='Train', color='royalblue', linewidth=2)
axes[0, 2].plot(epochs_imp2, df_imp2['val/dfl_loss'], label='Val', color='darkorange', linestyle='--', linewidth=2)
axes[0, 2].set_title('Distribution Focal Loss (DFL)')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Loss')
axes[0, 2].legend()

# mAP@0.50 plot
axes[1, 0].plot(epochs_imp2, df_imp2['metrics/mAP50(B)'], color='royalblue', linewidth=2)
axes[1, 0].set_title('mAP@0.50')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('mAP')

# mAP@0.50:0.95 plot
axes[1, 1].plot(epochs_imp2, df_imp2['metrics/mAP50-95(B)'], color='darkorange', linewidth=2)
axes[1, 1].set_title('mAP@0.50:0.95')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('mAP')

# precision & recall plot
axes[1, 2].plot(epochs_imp2, df_imp2['metrics/precision(B)'], label='Precision', color='mediumseagreen', linewidth=2)
axes[1, 2].plot(epochs_imp2, df_imp2['metrics/recall(B)'], label='Recall', color='dodgerblue', linewidth=2)
axes[1, 2].set_title('Precision & Recall')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('Score')
axes[1, 2].legend()

for ax in axes.flat:
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

fig.savefig(RUNS_DIR / IMP_NAME2 / 'comparison_loss_curves.png', bbox_inches='tight')
plt.show()
print('Saved comparison_loss_curves.png')

In [ ]:
'''
Plots composite loss for the experimental and improvement models
side-by-side for comparison
'''

# calculate composite loss for the improved model
df_imp2['train_total_loss'] = df_imp2[[c for c in [train_box_col, train_cls_col, train_dfl_col] if c]].sum(axis=1)
df_imp2['val_total_loss']   = df_imp2[[c for c in [val_box_col,   val_cls_col,   val_dfl_col]   if c]].sum(axis=1)
epochs_imp2 = df_imp2[epoch_col].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=FIG_DPI)
fig.suptitle('Improvement Cycle 1 vs. Cycle 2: Composite Training & Validation Loss', fontsize=13, fontweight='bold')

for ax, df_, epochs_, title in [
    (axes[0], df_imp, epochs_imp, 'Improvement Cycle 1'),
    (axes[1], df_imp2, epochs_imp2, 'Improvement Cycle 2'),
]:
    ax.plot(epochs_, df_['train_total_loss'], color='royalblue',  linewidth=2, label='Train')
    ax.plot(epochs_, df_['val_total_loss'],   color='darkorange', linewidth=2, linestyle='--', label='Val')

    # Annotate the best mAP@0.50 epoch for this model
    if map50_col and map50_col in df_.columns:
        best_idx_  = df_[map50_col].idxmax()
        best_ep_   = int(df_.loc[best_idx_, epoch_col])
        ax.axvline(best_ep_, color='#E2E535', linestyle=':', linewidth=1.5, alpha=0.9,
                   label=f'Best mAP@0.5 (epoch {best_ep_})')

    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Total Loss')
    ax.legend(); ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
fig.savefig(RUNS_DIR / IMP_NAME2 / 'imp1_vs_imp2_loss_curves.png', bbox_inches='tight')
plt.show()
print('Saved imp1_vs_imp2_loss_curves.png')

In [ ]:
'''
Plots mAP@0.50, mAP@0.50:0.95, and validation loss for
the experimental and improvement models side-by-side
for comparison.
'''
fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=FIG_DPI)
fig.suptitle('Improvement Cycle 1 vs. 2: mAP and Validation Loss', fontsize=13, fontweight='bold')

# draw the mAP@0.5 curves
ax = axes[0]
if map50_col:
    ax.plot(epochs_imp, df_imp[map50_col], color='royalblue', linewidth=2, label='Improvement Cycle 1')
    ax.plot(epochs_imp2, df_imp2[map50_col], color='darkorange', linewidth=2, linestyle='--', label='Improvement Cycle 2')

ax.set_title('mAP@0.5')
ax.set_xlabel('Epoch'); ax.set_ylabel('mAP')
ax.legend(); ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# draw the mAP@0.5:0.95 curves
ax = axes[1]
if map50_95_col:
    ax.plot(epochs_imp, df_imp[map50_95_col], color='royalblue', linewidth=2, label='Improvement Cycle 1')
    ax.plot(epochs_imp2, df_imp2[map50_95_col], color='darkorange', linewidth=2, linestyle='--', label='Improvement Cycle 2')

ax.set_title('mAP@0.5:0.95')
ax.set_xlabel('Epoch'); ax.set_ylabel('mAP')
ax.legend(); ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# draw the validation loss curves
ax = axes[2]
ax.plot(epochs_imp, df_imp['val_total_loss'], color='royalblue', linewidth=2, label='Improvement Cycle 1')
ax.plot(epochs_imp2, df_imp2['val_total_loss'], color='darkorange', linewidth=2, linestyle='--', label='Improvement Cycle 2')

ax.set_title('Validation Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Total Loss')
ax.legend(); ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
fig.savefig(RUNS_DIR / IMP_NAME2 / 'imp1_vs_imp2_metrics.png', bbox_inches='tight')
plt.show()
print('Saved imp1_vs_imp2_metrics.png')